# Logistic Regression Experiment - Final Validated Version

**Project:** How Does Graph Construction Influence the Effectiveness of Weisfeiler-Lehman Graph Kernels for Fine-Grained Plant-Stress Classification from Leaf Images?

**Task:** 57-class OLID-I plant stress classification  
**Model:** Logistic Regression (classical linear baseline)  
**Notebook:** `New_Models_Baselines/01_Logistic_Regression_Experiment_Final.ipynb`  
**Provenance:** 5-fold stratified cross-validation on 4,387 accepted samples (identical fold splits and features as classical baselines in NB-18)

---
## Summary of Audit and Final Targeted Corrections

| # | Topic | Original Issue | Correction Applied in Final Version |
|---|-------|----------------|-------------------------------------|
| 1 | **Estimator Lifecycle** | Single `LogisticRegression()` reused across folds | Fresh estimator instantiated inside each fold loop |
| 2 | **Timer Precision** | `time.time()` used | `time.perf_counter()` used throughout |
| 3 | **Convergence Monitoring** | Convergence warnings silenced via filter | Captured per fold, recorded in config JSON, reported in summary |
| 4 | **Data Validation** | No pre-training checks; unconditional pass | 9-point checklist with status accumulator; fails or warns explicitly |
| 5 | **Timing Terminology** | Full-batch prediction called "single-image latency" | Renamed to **"amortized batch prediction time per sample"** |
| 6 | **Single-Sample Benchmark** | No isolated single-sample measurement | Added dedicated warmed-up, repeated single-sample benchmark |
| 7 | **Standard Deviation** | `std()` ddof undocumented | Explicit sample standard deviation (`ddof=1`) from raw fold scores |
| 8 | **Configuration Record** | No machine-readable config | Saved comprehensive `logistic_regression_config.json` |
| 9 | **Directory Structure** | Output saved to root `.` | Saved to standardized `New_Models_Baselines/results/` and `figures/` |
| 10 | **Fold-Level Visualizations** | Mean-only plots | 5 figures generated including fold-level F1, accuracy, train time, latency, comparison |
| 11 | **Baseline Provenance** | Hardcoded baseline values without sources | Loaded from verified source CSVs (NB-18, NB-14) with source tracking |
| 12 | **Scientific Claims** | Unsupported causal claims (Fold 3, non-linearity) | Separated empirical observations from hypotheses; causal claims removed |


In [1]:
import os
import sys
import time
import json
import warnings as _warnings_module
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.exceptions import ConvergenceWarning

sw_versions = {
    "python":       sys.version.split()[0],
    "scikit-learn": sklearn.__version__,
    "numpy":        np.__version__,
    "pandas":       pd.__version__,
}
print("Software versions:")
for k, v in sw_versions.items():
    print("  {:20s}: {}".format(k, v))


Software versions:
  python              : 3.11.9
  scikit-learn        : 1.5.0
  numpy               : 1.26.4
  pandas              : 2.3.3


In [2]:
# ── Configuration ──────────────────────────────────────────────────────────
# All paths derived relative to workspace root so the notebook is self-contained.
cwd = os.getcwd()
if os.path.basename(cwd) == "New_Models_Baselines":
    WORKSPACE = os.path.dirname(cwd)
elif os.path.isdir(os.path.join(cwd, "New_Models_Baselines")):
    WORKSPACE = cwd
else:
    p = cwd
    WORKSPACE = cwd
    while p and os.path.dirname(p) != p:
        if os.path.isdir(os.path.join(p, "New_Models_Baselines")):
            WORKSPACE = p
            break
        p = os.path.dirname(p)

DATA_DIR  = os.path.join(WORKSPACE, "full_pipeline")
OUT_DIR   = os.path.join(WORKSPACE, "New_Models_Baselines", "results")
FIG_DIR   = os.path.join(WORKSPACE, "New_Models_Baselines", "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

FEAT_CSV = os.path.join(DATA_DIR, "olid_full_features.csv")
FOLD_CSV = os.path.join(DATA_DIR, "cv_fold_assignments.csv")
BL_SUM   = os.path.join(WORKSPACE, "full_pipeline", "results", "classical_baselines",
                         "classical_baselines_summary.csv")
WL_SUM   = os.path.join(WORKSPACE, "full_pipeline", "results", "wl_svm_classification",
                         "wl_svm_summary.csv")

# ── Experiment hyperparameters ────────────────────────────────────────────
RANDOM_STATE          = 42   # fixed seed for reproducibility
N_FOLDS               = 5    # five-fold stratified CV (pre-assigned splits)
N_TIMING_REPS         = 20   # repeated batch prediction calls for amortized latency
N_SINGLE_TIMING_REPS  = 50   # repeated single-sample calls for dedicated benchmark

# ── Model parameters (explicitly documented) ──────────────────────────────
LR_PARAMS = {
    "solver":       "lbfgs",   # LBFGS: memory-efficient, supports multinomial loss
    "max_iter":     1000,      # sufficient for convergence on 38-feature task
    "class_weight": "balanced",# balances unequal class frequencies (matches NB-18)
    "random_state": RANDOM_STATE,
    "n_jobs":       -1,
    "multi_class":  "auto",    # multinomial loss with lbfgs solver
}

print("Configuration loaded:")
print("  RANDOM_STATE         :", RANDOM_STATE)
print("  N_FOLDS              :", N_FOLDS)
print("  N_TIMING_REPS        :", N_TIMING_REPS, "(amortized batch prediction)")
print("  N_SINGLE_TIMING_REPS :", N_SINGLE_TIMING_REPS, "(dedicated single-sample benchmark)")
print("  LR_PARAMS            :", LR_PARAMS)
print("  OUT_DIR              :", OUT_DIR)
print("  FIG_DIR              :", FIG_DIR)


Configuration loaded:
  RANDOM_STATE         : 42
  N_FOLDS              : 5
  N_TIMING_REPS        : 20 (amortized batch prediction)
  N_SINGLE_TIMING_REPS : 50 (dedicated single-sample benchmark)
  LR_PARAMS            : {'solver': 'lbfgs', 'max_iter': 1000, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1, 'multi_class': 'auto'}
  OUT_DIR              : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\results
  FIG_DIR              : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 2 - DATA VALIDATION (9-point checklist with status accumulator)
# ════════════════════════════════════════════════════════════════════════════

assert os.path.isfile(FEAT_CSV), "Feature CSV not found: " + FEAT_CSV
assert os.path.isfile(FOLD_CSV), "Fold CSV not found: " + FOLD_CSV

df_full = pd.read_csv(FEAT_CSV)
df_acc  = df_full[df_full["quality_flag"] == "ACCEPT"].copy().reset_index(drop=True)

META_COLS = [
    "image_path", "image_name", "class", "crop", "condition",
    "quality_flag", "mask_area_ratio", "failure_reason"
]
# Derive FEATURE_COLS BEFORE the fold merge to prevent 'fold' entering features
FEATURE_COLS = [c for c in df_acc.columns if c not in META_COLS]

df_folds = pd.read_csv(FOLD_CSV)
df_acc   = df_acc.merge(df_folds[["image_path", "fold"]], on="image_path", how="left")

val_checks = []  # list of (check_id, check_name, status, detail)

# Check 1: Row count
ok1 = len(df_acc) == 4387
val_checks.append((1, "Row count", "PASS" if ok1 else "FAIL", "{} rows (expected 4387)".format(len(df_acc))))
assert ok1, "Expected 4387 rows, got " + str(len(df_acc))

# Check 2: Feature columns
ok2 = len(FEATURE_COLS) == 38
val_checks.append((2, "Feature columns", "PASS" if ok2 else "FAIL", "{} features (expected 38)".format(len(FEATURE_COLS))))
assert ok2, "Expected 38 feature columns, got " + str(len(FEATURE_COLS))

# Check 3: Target classes
n_classes = df_acc["class"].nunique()
ok3 = n_classes == 57
val_checks.append((3, "Target classes", "PASS" if ok3 else "FAIL", "{} unique classes (expected 57)".format(n_classes)))
assert ok3, "Expected 57 target classes, got " + str(n_classes)

# Check 4: NaN in features
nan_total = df_acc[FEATURE_COLS].isna().sum().sum()
ok4 = nan_total == 0
val_checks.append((4, "NaN in features", "PASS" if ok4 else "FAIL", "{} NaNs detected".format(nan_total)))
assert ok4, "Found NaN values in feature matrix"

# Check 5: Inf in features
inf_total = int(np.isinf(df_acc[FEATURE_COLS].values).sum())
ok5 = inf_total == 0
val_checks.append((5, "Inf in features", "PASS" if ok5 else "FAIL", "{} Infs detected".format(inf_total)))
assert ok5, "Found Inf values in feature matrix"

# Check 6: Unique images
n_unique = df_acc["image_path"].nunique()
ok6 = n_unique == len(df_acc)
val_checks.append((6, "Unique images", "PASS" if ok6 else "FAIL", "{} unique of {} rows".format(n_unique, len(df_acc))))
assert ok6, "Duplicate image paths detected in dataset"

# Check 7: Missing fold assignments
missing_folds = int(df_acc["fold"].isna().sum())
ok7 = missing_folds == 0
val_checks.append((7, "Missing folds", "PASS" if ok7 else "FAIL", "{} missing assignments".format(missing_folds)))
assert ok7, "Unassigned folds detected"

# Check 8: All 5 folds present
fold_counts = df_acc["fold"].value_counts().sort_index()
ok8 = set(fold_counts.index.astype(int)) == {0, 1, 2, 3, 4}
val_checks.append((8, "Fold partition sizes", "PASS" if ok8 else "FAIL", str(fold_counts.to_dict())))
assert ok8, "Expected folds 0..4"

# Check 9: Stratification (KL divergence per fold vs overall dist)
overall_dist = df_acc["class"].value_counts(normalize=True)
max_kl = 0.0
for fid in range(N_FOLDS):
    fd = (df_acc[df_acc["fold"] == fid]["class"]
          .value_counts(normalize=True)
          .reindex(overall_dist.index, fill_value=0.0))
    kl = float((overall_dist * np.log((overall_dist + 1e-9) / (fd + 1e-9))).sum())
    max_kl = max(max_kl, kl)
strat_ok = max_kl < 0.05
val_checks.append((9, "Stratification (KL)", "PASS" if strat_ok else "WARN",
                   "Max KL={:.6f} ({})".format(max_kl, "well-stratified" if strat_ok else "potential imbalance")))

# Print checklist
print("Data Validation Checklist:")
for cid, cname, cstatus, cdetail in val_checks:
    print("  [{:4s}] Check {} - {:24s}: {}".format(cstatus, cid, cname, cdetail))

# Determine overall validation status explicitly
n_fail = sum(1 for c in val_checks if c[2] == "FAIL")
n_warn = sum(1 for c in val_checks if c[2] == "WARN")
n_pass = sum(1 for c in val_checks if c[2] == "PASS")

print()
if n_fail > 0:
    print("DATA VALIDATION STATUS: FAILED ({} failed, {} warnings, {} passed)".format(n_fail, n_warn, n_pass))
    raise RuntimeError("Data validation failed with {} errors.".format(n_fail))
elif n_warn > 0:
    print("DATA VALIDATION STATUS: PASSED WITH WARNINGS ({} warning(s), {} passed clean)".format(n_warn, n_pass))
else:
    print("DATA VALIDATION STATUS: ALL CHECKS PASSED ({}/{} checks verified clean)".format(n_pass, len(val_checks)))


Data Validation Checklist:
  [PASS] Check 1 - Row count               : 4387 rows (expected 4387)
  [PASS] Check 2 - Feature columns         : 38 features (expected 38)
  [PASS] Check 3 - Target classes          : 57 unique classes (expected 57)
  [PASS] Check 4 - NaN in features         : 0 NaNs detected
  [PASS] Check 5 - Inf in features         : 0 Infs detected
  [PASS] Check 6 - Unique images           : 4387 unique of 4387 rows
  [PASS] Check 7 - Missing folds           : 0 missing assignments
  [PASS] Check 8 - Fold partition sizes    : {0: 878, 1: 878, 2: 877, 3: 877, 4: 877}
  [PASS] Check 9 - Stratification (KL)     : Max KL=0.000631 (well-stratified)

DATA VALIDATION STATUS: ALL CHECKS PASSED (9/9 checks verified clean)


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# SECTIONS 3 & 4 - LOGISTIC REGRESSION 5-FOLD CROSS-VALIDATION
# Fresh estimator per fold; preprocessing fitted on training fold only.
# Timing terminology: Amortized batch prediction time per sample.
# ════════════════════════════════════════════════════════════════════════════

class FoldSafeScaler:
    """
    Wraps StandardScaler to guarantee training-fold-only fitting.
    Fitted exclusively on the training partition; validation partition
    is transformed using training statistics only. Prevents data leakage.
    """
    def __init__(self):
        self._scaler = StandardScaler()
        self._fitted = False

    def fit_transform(self, X_df: pd.DataFrame) -> np.ndarray:
        arr = self._scaler.fit_transform(X_df.values)
        self._fitted = True
        return arr

    def transform(self, X_df: pd.DataFrame) -> np.ndarray:
        if not self._fitted:
            raise ValueError("FoldSafeScaler must be fitted before transform().")
        return self._scaler.transform(X_df.values)


fold_results  = []
conv_warnings = {}
fitted_models = {}  # retained for single-sample benchmark

print("Starting 5-fold cross-validation...")
print("  Fresh LogisticRegression estimator created for each fold.")
print("  Preprocessing fitted on training fold only.")
print("  Convergence warnings captured per fold (not silenced).")
print("  Timing metric: Amortized batch prediction time per sample ({} reps, median+IQR)".format(N_TIMING_REPS))
print()

for fid in range(N_FOLDS):
    # ── Split ────────────────────────────────────────────────────────────
    tr_idx = df_acc[df_acc["fold"] != fid].index
    va_idx = df_acc[df_acc["fold"] == fid].index

    X_tr = df_acc.loc[tr_idx, FEATURE_COLS]
    X_va = df_acc.loc[va_idx, FEATURE_COLS]
    y_tr = df_acc.loc[tr_idx, "class"].values
    y_va = df_acc.loc[va_idx, "class"].values
    n_val = len(y_va)

    # ── Fold-safe scaling ────────────────────────────────────────────────
    sc     = FoldSafeScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)

    # ── Fresh estimator (one per fold) ──────────────────────────────────
    model = LogisticRegression(**LR_PARAMS)

    # ── Training with convergence warning capture ────────────────────────
    cw = []
    with _warnings_module.catch_warnings(record=True) as caught:
        _warnings_module.simplefilter("always", ConvergenceWarning)
        t_train_start = time.perf_counter()
        model.fit(X_tr_s, y_tr)
        t_train_end   = time.perf_counter()
        cw = [str(w.message) for w in caught
              if issubclass(w.category, ConvergenceWarning)]
    t_train = t_train_end - t_train_start
    conv_warnings[fid] = cw
    fitted_models[fid] = (model, X_va_s, y_va)

    # ── Inference timing: Amortized batch prediction time per sample ─────
    # Warm-up call (primes JIT/caches)
    _ = model.predict(X_va_s[:1])

    # Single batch prediction for evaluation metrics
    tb_start = time.perf_counter()
    y_pred   = model.predict(X_va_s)
    tb_end   = time.perf_counter()
    t_batch  = tb_end - tb_start

    # Repeated full-validation-batch calls divided by n_val
    reps_batch = []
    for _ in range(N_TIMING_REPS):
        r0 = time.perf_counter()
        model.predict(X_va_s)
        r1 = time.perf_counter()
        reps_batch.append(r1 - r0)
    reps_batch = np.array(reps_batch)
    med_batch_s = float(np.median(reps_batch))
    iqr_batch_s = float(np.percentile(reps_batch, 75) - np.percentile(reps_batch, 25))
    
    # Amortized batch prediction time per sample (ms)
    amortized_batch_ms     = med_batch_s / n_val * 1000.0
    amortized_batch_iqr_ms = iqr_batch_s / n_val * 1000.0

    # ── Metrics ──────────────────────────────────────────────────────────
    acc = accuracy_score(y_va, y_pred)
    mf1 = f1_score(y_va, y_pred, average="macro",    zero_division=0)
    wf1 = f1_score(y_va, y_pred, average="weighted", zero_division=0)

    fold_results.append({
        "model":                             "Logistic Regression",
        "fold":                              fid,
        "accuracy":                          acc,
        "macro_f1":                          mf1,
        "weighted_f1":                       wf1,
        "train_time_sec":                    t_train,
        "prediction_batch_time_sec":         t_batch,
        "amortized_batch_ms_per_sample":     amortized_batch_ms,
        "amortized_batch_iqr_ms":            amortized_batch_iqr_ms,
        "median_latency_ms_per_sample":      amortized_batch_ms,  # alias for backward compatibility
        "inference_latency_iqr_ms":          amortized_batch_iqr_ms,  # alias for backward compatibility
        "n_val_samples":                     n_val,
        "n_convergence_warnings":            len(cw),
    })

    cw_flag = "CW:{}".format(len(cw)) if cw else "OK"
    print("  fold {}: acc={:.4f} mf1={:.4f} wf1={:.4f} "
          "train={:.3f}s batch={:.5f}s amortized={:.6f}ms/s iqr={:.6f} {}".format(
              fid, acc, mf1, wf1, t_train, t_batch, amortized_batch_ms, amortized_batch_iqr_ms, cw_flag))

print()
print("Convergence warnings per fold (max_iter=1000):")
any_cw = any(len(msgs) > 0 for msgs in conv_warnings.values())
for fid, msgs in conv_warnings.items():
    if msgs:
        print("  fold {}: WARN - {} message(s)".format(fid, len(msgs)))
        for m in msgs:
            print("    ", m)
    else:
        print("  fold {}: OK - converged".format(fid))
if any_cw:
    print("  ACTION: Convergence warnings detected - consider increasing max_iter.")
else:
    print("  All folds converged within max_iter=1000.")


Starting 5-fold cross-validation...
  Fresh LogisticRegression estimator created for each fold.
  Preprocessing fitted on training fold only.
  Convergence warnings captured per fold (not silenced).
  Timing metric: Amortized batch prediction time per sample (20 reps, median+IQR)



  fold 0: acc=0.4522 mf1=0.4253 wf1=0.4582 train=2.052s batch=0.00147s amortized=0.000730ms/s iqr=0.000414 OK


  fold 1: acc=0.4214 mf1=0.3993 wf1=0.4219 train=1.363s batch=0.00208s amortized=0.001068ms/s iqr=0.000496 OK


  fold 2: acc=0.4162 mf1=0.3886 wf1=0.4178 train=1.423s batch=0.00185s amortized=0.001169ms/s iqr=0.000231 OK


  fold 3: acc=0.4903 mf1=0.4657 wf1=0.4896 train=1.481s batch=0.00084s amortized=0.001059ms/s iqr=0.000230 OK


  fold 4: acc=0.4458 mf1=0.4243 wf1=0.4452 train=1.478s batch=0.00197s amortized=0.001603ms/s iqr=0.000532 OK

Convergence warnings per fold (max_iter=1000):
  fold 0: OK - converged
  fold 1: OK - converged
  fold 2: OK - converged
  fold 3: OK - converged
  fold 4: OK - converged
  All folds converged within max_iter=1000.


In [5]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 4B - DEDICATED SINGLE-SAMPLE PREDICTION BENCHMARK
# Measures isolated, unbatched single-sample query latency after warm-up.
# Contrasted with amortized batch prediction time per sample.
# ════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("DEDICATED SINGLE-SAMPLE PREDICTION BENCHMARK")
print("=" * 70)
print("Protocol:")
print("  - Isolated prediction of single samples (shape: 1 x 38)")
print("  - 10 warm-up prediction calls (discarded) to prime CPU caches")
print("  - {} timed repetitions per fold using time.perf_counter()".format(N_SINGLE_TIMING_REPS))
print()

single_benchmarks = []
for fid in range(N_FOLDS):
    model, X_va_s, _ = fitted_models[fid]
    x_single = X_va_s[0:1]  # single test sample
    
    # Warm-up calls
    for _ in range(10):
        _ = model.predict(x_single)
        
    # Timed isolated single-sample predictions
    single_reps = []
    for _ in range(N_SINGLE_TIMING_REPS):
        t0 = time.perf_counter()
        _ = model.predict(x_single)
        t1 = time.perf_counter()
        single_reps.append(t1 - t0)
    single_reps = np.array(single_reps) * 1000.0  # convert to ms
    
    med_single = float(np.median(single_reps))
    iqr_single = float(np.percentile(single_reps, 75) - np.percentile(single_reps, 25))
    single_benchmarks.append({"fold": fid, "single_sample_median_ms": med_single, "single_sample_iqr_ms": iqr_single})
    print("  fold {}: single-sample median = {:.4f} ms, IQR = {:.4f} ms".format(
        fid, med_single, iqr_single))

df_single = pd.DataFrame(single_benchmarks)
mean_single_ms  = float(df_single["single_sample_median_ms"].mean())
mean_single_iqr = float(df_single["single_sample_iqr_ms"].mean())

# Add single-sample metrics to fold_results
for i in range(len(fold_results)):
    fold_results[i]["single_sample_median_ms"] = single_benchmarks[i]["single_sample_median_ms"]
    fold_results[i]["single_sample_iqr_ms"]    = single_benchmarks[i]["single_sample_iqr_ms"]

print()
print("Inference Timing Comparison (Model-Only, 38 Scaled Features):")
print("  1. Dedicated Single-Sample Prediction : {:.4f} ms (median over isolated calls)".format(mean_single_ms))
print("     Single-Sample IQR                  : {:.4f} ms".format(mean_single_iqr))
print("  2. Amortized Batch Prediction / Sample: {:.6f} ms (batch time divided by n_val)".format(
    float(pd.DataFrame(fold_results)["amortized_batch_ms_per_sample"].mean())))
print("     Ratio (Single / Amortized)         : ~{:.1f}x slower per query".format(
    mean_single_ms / float(pd.DataFrame(fold_results)["amortized_batch_ms_per_sample"].mean())))
print()
print("Architectural Rationale:")
print("  - Batch prediction exploits vectorized BLAS matrix multiplication in C/Fortran,")
print("    amortizing Python dispatch overhead across the entire validation batch (~878 samples).")
print("  - Single-sample prediction incurs per-call Python function invocation, input validation,")
print("    and array boxing overhead for each isolated query.")
print("  - Neither metric includes leaf image segmentation, graph construction, or feature extraction.")
print("=" * 70)


DEDICATED SINGLE-SAMPLE PREDICTION BENCHMARK
Protocol:
  - Isolated prediction of single samples (shape: 1 x 38)
  - 10 warm-up prediction calls (discarded) to prime CPU caches
  - 50 timed repetitions per fold using time.perf_counter()

  fold 0: single-sample median = 0.0449 ms, IQR = 0.0016 ms
  fold 1: single-sample median = 0.0452 ms, IQR = 0.0016 ms
  fold 2: single-sample median = 0.0436 ms, IQR = 0.0013 ms
  fold 3: single-sample median = 0.0362 ms, IQR = 0.0025 ms
  fold 4: single-sample median = 0.0364 ms, IQR = 0.0029 ms

Inference Timing Comparison (Model-Only, 38 Scaled Features):
  1. Dedicated Single-Sample Prediction : 0.0413 ms (median over isolated calls)
     Single-Sample IQR                  : 0.0020 ms
  2. Amortized Batch Prediction / Sample: 0.001126 ms (batch time divided by n_val)
     Ratio (Single / Amortized)         : ~36.6x slower per query

Architectural Rationale:
  - Batch prediction exploits vectorized BLAS matrix multiplication in C/Fortran,
    amor

In [6]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 5 - SUMMARY STATISTICS
# Standard deviation uses ddof=1 (sample std).
# Computed from raw fold-level values, not from rounded figures.
# ════════════════════════════════════════════════════════════════════════════

df_fold = pd.DataFrame(fold_results)

ma  = df_fold["accuracy"].mean()
sa  = df_fold["accuracy"].std(ddof=1)
mm  = df_fold["macro_f1"].mean()
sm  = df_fold["macro_f1"].std(ddof=1)
mw  = df_fold["weighted_f1"].mean()
sw2 = df_fold["weighted_f1"].std(ddof=1)
mt  = df_fold["train_time_sec"].mean()
mb  = df_fold["prediction_batch_time_sec"].mean()
ml  = df_fold["amortized_batch_ms_per_sample"].mean()
mi  = df_fold["amortized_batch_iqr_ms"].mean()
mss = df_fold["single_sample_median_ms"].mean()
msi = df_fold["single_sample_iqr_ms"].mean()

print("Fold-level results:")
print(df_fold[["fold", "accuracy", "macro_f1", "weighted_f1",
               "train_time_sec", "amortized_batch_ms_per_sample",
               "single_sample_median_ms"]].to_string(index=False))
print()
print("=" * 70)
print("Mean +/- SD (ddof=1, computed from raw fold values):")
print("  Accuracy    : {:.6f} +/- {:.6f}".format(ma, sa))
print("  Macro-F1    : {:.6f} +/- {:.6f}  <-- primary metric".format(mm, sm))
print("  Weighted-F1 : {:.6f} +/- {:.6f}".format(mw, sw2))
print()
print("Timing Breakdown (model-only; features pre-extracted and pre-scaled):")
print("  Mean train time                     : {:.4f} s".format(mt))
print("  Mean batch pred time (full fold)    : {:.6f} s".format(mb))
print("  Mean amortized batch pred per sample: {:.6f} ms/sample ({} reps per fold)".format(ml, N_TIMING_REPS))
print("  Mean amortized batch IQR per sample : {:.6f} ms/sample".format(mi))
print("  Mean single-sample pred latency     : {:.4f} ms/sample ({} reps per fold)".format(mss, N_SINGLE_TIMING_REPS))
print("  Mean single-sample pred IQR         : {:.4f} ms/sample".format(msi))
print()
print("CAUTION: Inference timings are model-only (features already extracted")
print("and scaled). They are NOT comparable with WL end-to-end latencies.")
print("=" * 70)


Fold-level results:
 fold  accuracy  macro_f1  weighted_f1  train_time_sec  amortized_batch_ms_per_sample  single_sample_median_ms
    0  0.452164  0.425268     0.458246        2.051664                       0.000730                  0.04485
    1  0.421412  0.399269     0.421937        1.362590                       0.001068                  0.04520
    2  0.416192  0.388636     0.417839        1.423044                       0.001169                  0.04360
    3  0.490308  0.465696     0.489582        1.481376                       0.001059                  0.03625
    4  0.445838  0.424329     0.445197        1.477905                       0.001603                  0.03640

Mean +/- SD (ddof=1, computed from raw fold values):
  Accuracy    : 0.445183 +/- 0.029542
  Macro-F1    : 0.420640 +/- 0.029774  <-- primary metric
  Weighted-F1 : 0.446560 +/- 0.029242

Timing Breakdown (model-only; features pre-extracted and pre-scaled):
  Mean train time                     : 1.5593 s
  Mean

In [7]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 6 - SAVE RESULTS & CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

fold_csv_path    = os.path.join(OUT_DIR, "logistic_regression_fold_results.csv")
summary_csv_path = os.path.join(OUT_DIR, "logistic_regression_summary.csv")
config_json_path = os.path.join(OUT_DIR, "logistic_regression_config.json")

df_fold.to_csv(fold_csv_path, index=False)

summary_row = {
    "model":                                  "Logistic Regression",
    "mean_accuracy":                          ma,
    "std_accuracy_ddof1":                     sa,
    "mean_macro_f1":                          mm,
    "std_macro_f1_ddof1":                     sm,
    "mean_weighted_f1":                       mw,
    "std_weighted_f1_ddof1":                  sw2,
    "mean_train_time_sec":                    mt,
    "mean_prediction_batch_time_sec":         mb,
    "mean_amortized_batch_ms_per_sample":     ml,
    "mean_amortized_batch_iqr_ms":            mi,
    "mean_median_latency_ms_per_sample":      ml,  # backward compatibility alias
    "mean_iqr_latency_ms":                    mi,  # backward compatibility alias
    "mean_single_sample_median_ms":           mss,
    "mean_single_sample_iqr_ms":              msi,
    "n_timing_repetitions_batch":             N_TIMING_REPS,
    "n_timing_repetitions_single":            N_SINGLE_TIMING_REPS,
    "timing_scope":                           "model-only (features pre-extracted and pre-scaled)",
    "std_ddof":                               1,
}
pd.DataFrame([summary_row]).to_csv(summary_csv_path, index=False)

config = {
    "experiment":   "Logistic Regression - 57-class OLID-I plant stress (Final Validated)",
    "data_source":  FEAT_CSV,
    "fold_source":  FOLD_CSV,
    "feature_cols": FEATURE_COLS,
    "target_col":   "class",
    "n_accepted":   int(len(df_acc)),
    "n_features":   int(len(FEATURE_COLS)),
    "n_classes":    57,
    "n_folds":      N_FOLDS,
    "model_params": LR_PARAMS,
    "preprocessing": "StandardScaler fitted on training fold only (FoldSafeScaler)",
    "timing": {
        "method":                      "time.perf_counter()",
        "train_scope":                 "model.fit() only",
        "inference_scope":             "model-only (no image/graph processing)",
        "warm_up":                     "Warm-up prediction calls discarded before timing",
        "amortized_batch_prediction": {
            "n_reps":                  N_TIMING_REPS,
            "metric":                  "Amortized batch prediction time per sample (median & IQR)",
            "description":             "Repeated validation batch predictions divided by validation size"
        },
        "single_sample_benchmark": {
            "n_reps":                  N_SINGLE_TIMING_REPS,
            "metric":                  "Dedicated isolated single-sample prediction latency (median & IQR)",
            "description":             "Repeated prediction of single unbatched sample (1 x 38) after warm-up"
        },
        "scope_caution":               "Model-only latencies; not comparable to end-to-end WL pipeline."
    },
    "std_ddof":    1,
    "random_seed": RANDOM_STATE,
    "sw_versions": sw_versions,
    "convergence_warnings": {str(k): v for k, v in conv_warnings.items()},
}
with open(config_json_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved fold results :", fold_csv_path)
print("Saved summary      :", summary_csv_path)
print("Saved config JSON  :", config_json_path)


Saved fold results : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\results\logistic_regression_fold_results.csv
Saved summary      : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\results\logistic_regression_summary.csv
Saved config JSON  : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\results\logistic_regression_config.json


In [8]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 7 - VERIFIED HISTORICAL BASELINES & PROVENANCE
# Historical values are loaded directly from source CSVs (NB-18, NB-14)
# and cross-checked against reference specification values.
# ════════════════════════════════════════════════════════════════════════════

df_bls = pd.read_csv(BL_SUM)   # NB-18 classical_baselines_summary.csv
df_wls = pd.read_csv(WL_SUM)   # NB-14 wl_svm_summary.csv

def get_baseline(df, method_col, method_val):
    row = df[df[method_col] == method_val]
    if len(row) == 0:
        return None, None
    return float(row["mean_accuracy"].iloc[0]), float(row["mean_macro_f1"].iloc[0])

lsvm_a, lsvm_m = get_baseline(df_bls, "method", "Linear SVM")
rbf_a,  rbf_m  = get_baseline(df_bls, "method", "RBF SVM (Ref)")
rf_a,   rf_m   = get_baseline(df_bls, "method", "Random Forest")
knn_a,  knn_m  = get_baseline(df_bls, "method", "KNN (k=5)")

wl2 = df_wls[(df_wls["task"] == "57-class") &
             (df_wls["graph_type"] == "knn") &
             (df_wls["wl_height"] == 2)]
wl_a = float(wl2["mean_accuracy"].iloc[0])
wl_m = float(wl2["mean_macro_f1"].iloc[0])

# Cross-check against task spec reference values
ref = {
    "Linear SVM":                  (0.4855, 0.4720),
    "RBF SVM":                     (0.4554, 0.4307),
    "Random Forest":               (0.5854, 0.5499),
    "k-NN":                        (0.4876, 0.4102),
    "Standard WL feature-kNN h=2": (0.4250, 0.3911),
}
verified = {
    "Linear SVM":                  (lsvm_a, lsvm_m),
    "RBF SVM":                     (rbf_a,  rbf_m),
    "Random Forest":               (rf_a,   rf_m),
    "k-NN":                        (knn_a,  knn_m),
    "Standard WL feature-kNN h=2": (wl_a,   wl_m),
}

print("Provenance Cross-Check: Reference vs. Verified Source CSVs:")
discrepancy = False
for name in ref:
    ra, rm = ref[name]
    va, vm = verified[name]
    ok = (va is not None and abs(ra - va) < 0.001 and abs(rm - vm) < 0.001)
    if not ok:
        discrepancy = True
    status = "MATCH" if ok else "DISCREPANCY"
    print("  {:40s} ref=({:.4f},{:.4f})  src=({:.4f},{:.4f})  {}".format(
        name, ra, rm, va, vm, status))

if discrepancy:
    print("  NOTE: Discrepancies detected - using verified source CSV values.")
else:
    print("  All reference values match source CSVs within tolerance (<0.001).")

# Comparison table distinguishing historical baselines from new LR measurement
comparison = [
    {"Method": "Linear SVM",               "Source": "NB-18 verified (classical_baselines_summary.csv)", "Accuracy": lsvm_a, "Macro-F1": lsvm_m},
    {"Method": "RBF SVM",                  "Source": "NB-18 verified (classical_baselines_summary.csv)", "Accuracy": rbf_a,  "Macro-F1": rbf_m},
    {"Method": "Random Forest",            "Source": "NB-18 verified (classical_baselines_summary.csv)", "Accuracy": rf_a,   "Macro-F1": rf_m},
    {"Method": "k-NN (k=5)",               "Source": "NB-18 verified (classical_baselines_summary.csv)", "Accuracy": knn_a,  "Macro-F1": knn_m},
    {"Method": "WL+SVM kNN-graph h=2",     "Source": "NB-14 verified (wl_svm_summary.csv)",              "Accuracy": wl_a,   "Macro-F1": wl_m},
    {"Method": "Logistic Regression (NEW)","Source": "This experiment (5-fold CV)",                      "Accuracy": ma,     "Macro-F1": mm},
]
df_cmp = pd.DataFrame(comparison)
cmp_path = os.path.join(OUT_DIR, "logistic_regression_comparison.csv")
df_cmp.to_csv(cmp_path, index=False)

print()
print("Model Comparison Table (Historical Provenance vs. Newly Measured):")
print(df_cmp.to_string(index=False))
print("Saved comparison table to:", cmp_path)


Provenance Cross-Check: Reference vs. Verified Source CSVs:
  Linear SVM                               ref=(0.4855,0.4720)  src=(0.4855,0.4720)  MATCH
  RBF SVM                                  ref=(0.4554,0.4307)  src=(0.4554,0.4307)  MATCH
  Random Forest                            ref=(0.5854,0.5499)  src=(0.5854,0.5499)  MATCH
  k-NN                                     ref=(0.4876,0.4102)  src=(0.4876,0.4102)  MATCH
  Standard WL feature-kNN h=2              ref=(0.4250,0.3911)  src=(0.4249,0.3911)  MATCH
  All reference values match source CSVs within tolerance (<0.001).

Model Comparison Table (Historical Provenance vs. Newly Measured):
                   Method                                           Source  Accuracy  Macro-F1
               Linear SVM NB-18 verified (classical_baselines_summary.csv)  0.485526  0.471995
                  RBF SVM NB-18 verified (classical_baselines_summary.csv)  0.455438  0.430721
            Random Forest NB-18 verified (classical_baselines_su

In [9]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 8 - VISUALIZATIONS
# ════════════════════════════════════════════════════════════════════════════

plt.rcParams.update({
    "font.size":      11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "figure.dpi":     150,
})

folds_x   = list(range(N_FOLDS))
mf1_vals  = df_fold["macro_f1"].tolist()
acc_vals  = df_fold["accuracy"].tolist()
tr_vals   = df_fold["train_time_sec"].tolist()
batch_lat = df_fold["amortized_batch_ms_per_sample"].tolist()
batch_iqr = df_fold["amortized_batch_iqr_ms"].tolist()

def save_fig(fig, name):
    p = os.path.join(FIG_DIR, name)
    fig.savefig(p, bbox_inches="tight")
    plt.close(fig)
    print("  Saved figure:", p)
    return p

# ── Figure 1: Fold-level Macro-F1 ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(folds_x, mf1_vals, marker="o", lw=2, color="#2563EB", label="Macro-F1 per fold")
ax.axhline(mm, color="#DC2626", ls="--", lw=1.5, label="Mean = {:.4f}".format(mm))
ax.fill_between(folds_x, [mm - sm]*N_FOLDS, [mm + sm]*N_FOLDS,
                alpha=0.15, color="#DC2626", label="+/-1 SD ({:.4f})".format(sm))
ax.set_xlabel("Fold"); ax.set_ylabel("Macro-F1")
ax.set_title("Logistic Regression - Macro-F1 per Fold\n(57-class OLID-I plant stress)")
ax.set_xticks(folds_x); ax.set_ylim(0, 0.7)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
save_fig(fig, "lr_macro_f1_folds.png")

# ── Figure 2: Fold-level Accuracy ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(folds_x, acc_vals, marker="s", lw=2, color="#059669", label="Accuracy per fold")
ax.axhline(ma, color="#D97706", ls="--", lw=1.5, label="Mean = {:.4f}".format(ma))
ax.fill_between(folds_x, [ma - sa]*N_FOLDS, [ma + sa]*N_FOLDS,
                alpha=0.15, color="#D97706", label="+/-1 SD ({:.4f})".format(sa))
ax.set_xlabel("Fold"); ax.set_ylabel("Accuracy")
ax.set_title("Logistic Regression - Accuracy per Fold\n(57-class OLID-I plant stress)")
ax.set_xticks(folds_x); ax.set_ylim(0, 0.7)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
save_fig(fig, "lr_accuracy_folds.png")

# ── Figure 3: Training time per fold ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(folds_x, tr_vals, color="#7C3AED", alpha=0.8)
ax.axhline(mt, color="#DC2626", ls="--", lw=1.5, label="Mean = {:.3f}s".format(mt))
for bar, t in zip(bars, tr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            "{:.2f}s".format(t), ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Fold"); ax.set_ylabel("Training Time (seconds)")
ax.set_title("Logistic Regression - Training Time per Fold")
ax.set_xticks(folds_x); ax.legend(fontsize=9); ax.grid(True, axis="y", alpha=0.3)
save_fig(fig, "lr_training_time.png")

# ── Figure 4: Amortized batch prediction time per fold (corrected terminology)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(folds_x, batch_lat, color="#0891B2", alpha=0.8, label="Amortized batch time (median)")
ax.errorbar(folds_x, batch_lat, yerr=batch_iqr, fmt="none",
            color="black", capsize=5, lw=1.5, label="IQR")
ax.set_xlabel("Fold"); ax.set_ylabel("Amortized Batch Time (ms / sample)")
ax.set_title("Logistic Regression - Amortized Batch Prediction Time per Fold\n"
             "(Repeated validation batch predictions divided by n_val; model-only)")
ax.set_xticks(folds_x); ax.legend(fontsize=9); ax.grid(True, axis="y", alpha=0.3)
save_fig(fig, "lr_inference_latency.png")

# ── Figure 5: Comparison with verified historical baselines ──────────────────
methods_cmp = [r["Method"] for r in comparison]
mf1_cmp     = [r["Macro-F1"] for r in comparison]
colors_cmp  = ["#6B7280"] * (len(methods_cmp) - 1) + ["#DC2626"]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(methods_cmp, mf1_cmp, color=colors_cmp, alpha=0.85)
for bar, val in zip(bars, mf1_cmp):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            "{:.4f}".format(val), va="center", ha="left", fontsize=9)
ax.errorbar(mm, len(methods_cmp) - 1, xerr=sm,
            fmt="none", color="black", capsize=5, lw=2,
            label="LR +/-1 SD ({:.4f})\n(fold-level SD available for LR only)".format(sm))
ax.set_xlabel("Macro-F1"); ax.set_xlim(0, 0.7)
ax.set_title("Macro-F1 Comparison - Logistic Regression vs. Verified Baselines\n"
             "(57-class OLID-I; historical baselines from NB-18 and NB-14)")
ax.axvline(mm, color="#DC2626", ls=":", alpha=0.5)
ax.legend(fontsize=8, loc="lower right"); ax.grid(True, axis="x", alpha=0.3)
save_fig(fig, "lr_comparison_macro_f1.png")


  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\lr_macro_f1_folds.png


  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\lr_accuracy_folds.png
  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\lr_training_time.png


  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\lr_inference_latency.png
  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\lr_comparison_macro_f1.png


'C:\\Users\\merug\\OneDrive\\Desktop\\FINAL_YEAR_PROJECT\\WL_Graph_Representation\\New_Models_Baselines\\figures\\lr_comparison_macro_f1.png'

In [10]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 9 - SCIENTIFIC INTERPRETATION
# Observations reported strictly separately from unverified hypotheses.
# ════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("SCIENTIFIC INTERPRETATION - EMPIRICAL FINDINGS & HYPOTHESES")
print("=" * 70)
print()
print("1. EVALUATION PERFORMANCE SUMMARY (57-class OLID-I):")
print("  - Macro-F1    : {:.4f} +/- {:.4f} (ddof=1)  [Primary Metric]".format(mm, sm))
print("  - Accuracy    : {:.4f} +/- {:.4f}".format(ma, sa))
print("  - Weighted-F1 : {:.4f} +/- {:.4f}".format(mw, sw2))
print()
print("2. COMPARATIVE RANKING (Macro-F1, Descending):")
for r in sorted(comparison, key=lambda x: x["Macro-F1"], reverse=True):
    marker = "  <-- THIS EXPERIMENT (NEW)" if r["Method"].startswith("Logistic") else "  [Historical baseline]"
    print("  {:35s} : {:.4f}  {}".format(r["Method"], r["Macro-F1"], marker))
print()
print("3. EMPIRICAL OBSERVATIONS vs. HYPOTHESES:")
print()
print("  Observation A (Performance Relative to Baselines):")
print("    - Logistic Regression (Macro-F1={:.4f}) ranks below Random Forest ({:.4f}),".format(mm, rf_m))
print("      Linear SVM ({:.4f}), and RBF SVM ({:.4f}), but above k-NN ({:.4f})".format(lsvm_m, rbf_m, knn_m))
print("      and standard WL+SVM kNN-graph h=2 ({:.4f}).".format(wl_m))
print("    - LR exceeds standard WL graph kernels by {:+.4f} Macro-F1, consistent with".format(mm - wl_m))
print("      prior project findings that 38 hand-crafted features outperform standard")
print("      graph topological representations on this dataset.")
print()
print("  Observation B & Hypothesis regarding the Random Forest Gap:")
print("    - Observation: Random Forest outperforms Logistic Regression by {:+.4f} Macro-F1.".format(rf_m - mm))
print("    - Hypothesis / Discussion: Logistic Regression uses a linear decision boundary in log-odds.")
print("      However, attributing the entire gap to 'non-linear separability' is an unverified hypothesis.")
print("      Random Forest also benefits from axis-aligned feature thresholding, interaction capture,")
print("      and ensemble variance reduction. Furthermore, Linear SVM (also a linear model) achieves")
print("      {:.4f} Macro-F1 ({:+.4f} higher than LR), demonstrating that margin formulation and loss".format(lsvm_m, lsvm_m - mm))
print("      function significantly impact performance on this feature space beyond decision boundary geometry alone.")
print()
print("  Observation C & Hypothesis regarding Cross-Fold Variation (Fold 3):")
print("    - Observation: Macro-F1 across the 5 folds spans [{:.4f}, {:.4f}] (mean={:.4f}, SD={:.4f}).".format(
    min(mf1_vals), max(mf1_vals), mm, sm))
print("      Fold 3 records the highest Macro-F1 ({:.4f}), which is {:+.4f} above the 5-fold mean.".format(
    mf1_vals[3], mf1_vals[3] - mm))
print("    - Hypothesis / Discussion: The root cause of Fold 3's elevated score is undetermined from")
print("      aggregate metrics alone. Plausible hypotheses include partition-specific test sample difficulty,")
print("      sub-class distribution variations, or stochastic CV partition noise. We state the elevated")
print("      performance as an empirical observation without claiming a verified causal explanation.")
print()
print("4. TIMING TERMINOLOGY & BENCHMARK DISTINCTION:")
print("  - Amortized Batch Prediction Time : {:.6f} ms/sample (IQR: {:.6f} ms)".format(ml, mi))
print("    (Repeated full-validation-batch predictions divided by validation size)")
print("  - Dedicated Single-Sample Latency : {:.4f} ms/sample (IQR: {:.4f} ms)".format(mss, msi))
print("    (Repeated isolated single-sample predictions after warm-up)")
print("  - Amortized batch inference is ~{:.1f}x faster per sample due to BLAS matrix vectorization,".format(mss / ml))
print("    whereas single-sample prediction includes per-call Python dispatch overhead.")
print("  - Scope reminder: Both measurements are model-only (pre-extracted features) and cannot")
print("    be compared against end-to-end WL pipeline runtimes.")
print()
print("5. METHODOLOGICAL INTEGRITY & CONVERGENCE:")
print("  - All 5 folds converged cleanly within max_iter=1000 (0 ConvergenceWarnings).")
print("  - Stratified 5-fold CV splits, 38 feature columns, and balanced class weights")
print("    were maintained identically to the verified historical protocol.")
print("=" * 70)


SCIENTIFIC INTERPRETATION - EMPIRICAL FINDINGS & HYPOTHESES

1. EVALUATION PERFORMANCE SUMMARY (57-class OLID-I):
  - Macro-F1    : 0.4206 +/- 0.0298 (ddof=1)  [Primary Metric]
  - Accuracy    : 0.4452 +/- 0.0295
  - Weighted-F1 : 0.4466 +/- 0.0292

2. COMPARATIVE RANKING (Macro-F1, Descending):
  Random Forest                       : 0.5499    [Historical baseline]
  Linear SVM                          : 0.4720    [Historical baseline]
  RBF SVM                             : 0.4307    [Historical baseline]
  Logistic Regression (NEW)           : 0.4206    <-- THIS EXPERIMENT (NEW)
  k-NN (k=5)                          : 0.4102    [Historical baseline]
  WL+SVM kNN-graph h=2                : 0.3911    [Historical baseline]

3. EMPIRICAL OBSERVATIONS vs. HYPOTHESES:

  Observation A (Performance Relative to Baselines):
    - Logistic Regression (Macro-F1=0.4206) ranks below Random Forest (0.5499),
      Linear SVM (0.4720), and RBF SVM (0.4307), but above k-NN (0.4102)
      and standar

In [11]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 10 - FINAL EXECUTION REPORT & ARTIFACT VERIFICATION
# ════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("FINAL EXECUTION REPORT & WORKSPACE VERIFICATION")
print("=" * 70)
print()

tracked_files = [
    os.path.join(OUT_DIR, "logistic_regression_fold_results.csv"),
    os.path.join(OUT_DIR, "logistic_regression_summary.csv"),
    os.path.join(OUT_DIR, "logistic_regression_config.json"),
    os.path.join(OUT_DIR, "logistic_regression_comparison.csv"),
    os.path.join(FIG_DIR, "lr_macro_f1_folds.png"),
    os.path.join(FIG_DIR, "lr_accuracy_folds.png"),
    os.path.join(FIG_DIR, "lr_training_time.png"),
    os.path.join(FIG_DIR, "lr_inference_latency.png"),
    os.path.join(FIG_DIR, "lr_comparison_macro_f1.png"),
]

print("Workspace File Verification:")
all_files_exist = True
for p in tracked_files:
    exists = os.path.isfile(p)
    if not exists:
        all_files_exist = False
    size_str = "{:>8,d} bytes".format(os.path.getsize(p)) if exists else "     MISSING"
    status = "[OK]     " if exists else "[MISSING]"
    print("  {} {:38s} : {}".format(status, os.path.basename(p), size_str))

print()
print("Validation Status Summary:")
print("  - Data validation checks : {}/{} checks passed cleanly".format(n_pass, len(val_checks)))
print("  - Convergence warnings   : {}".format("None (all folds converged)" if not any_cw else "Warnings detected"))
print("  - All artifacts verified : {}".format("YES" if all_files_exist else "NO"))
print()
print("Status: READY FOR REVIEW")
print()
print("NOTE: Stop here. Do not implement the next model.")
print("=" * 70)


FINAL EXECUTION REPORT & WORKSPACE VERIFICATION

Workspace File Verification:
  [OK]      logistic_regression_fold_results.csv   :    1,555 bytes
  [OK]      logistic_regression_summary.csv        :      784 bytes
  [OK]      logistic_regression_config.json        :    2,523 bytes
  [OK]      logistic_regression_comparison.csv     :      617 bytes
  [OK]      lr_macro_f1_folds.png                  :   53,644 bytes
  [OK]      lr_accuracy_folds.png                  :   53,723 bytes
  [OK]      lr_training_time.png                   :   45,596 bytes
  [OK]      lr_inference_latency.png               :   70,036 bytes
  [OK]      lr_comparison_macro_f1.png             :   85,533 bytes

Validation Status Summary:
  - Data validation checks : 9/9 checks passed cleanly
  - Convergence warnings   : None (all folds converged)
  - All artifacts verified : YES

Status: READY FOR REVIEW

NOTE: Stop here. Do not implement the next model.
